# 02 — LoRA Fine-tuning + Push to HuggingFace

**Run on Modal GPU notebook (A100 recommended).**

Steps:
1. Pull patched Moshi weights from your private HF repo
2. Apply LoRA to temporal transformer attention layers (PEFT)
3. Fine-tune on `data/tool_calls.jsonl` — text stream only
4. Merge LoRA and push fine-tuned model to HuggingFace

VRAM guide:
| GPU | VRAM | LoRA rank | Batch |
|-----|------|-----------|-------|
| A100 40/80 GB | 40-80 GB | 16 | 4-8 |
| RTX 3090/4090 | 24 GB | 8 | 2 |
| V100 16 GB | 16 GB | 4 | 1 |

In [ ]:
import subprocess, torch
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout.strip())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

In [ ]:
!pip install peft==0.11.0 safetensors sentencepiece huggingface_hub einops sphn aiohttp numpy -q

In [ ]:
import gc, json, os, shutil, subprocess, sys
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO = Path("/repo")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone",
    "https://github.com/syedfahimabrar/moshi-D-gu.git",
    str(REPO)], check=True)
sys.path.insert(0, str(REPO / "moshi"))

import sentencepiece as spm
from moshi.models.loaders import get_moshi_lm

# ── Config ────────────────────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"
HF_OUTPUT_REPO  = "abrarfahim/moshi-tool-finetuned"
HF_TOKEN        = os.environ.get("HF_TOKEN", None)  # or paste token here

LORA_RANK    = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
BATCH_SIZE   = 2     # reduce to 1 if OOM
GRAD_ACCUM   = 4
LR           = 2e-4
EPOCHS       = 10
MAX_SEQ_LEN  = 512

WEIGHTS_DIR = Path("/tmp/moshi_weights")
OUT_DIR     = Path("/mnt/model")
DATA_PATH   = REPO / "notebooks/data/tool_calls.jsonl"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE       = torch.bfloat16

PAD_ID             = 3
TOOL_CALL_ID       = 32000
TOOL_END_ID        = 32001
TOOL_RESULT_ID     = 32002
TOOL_RESULT_END_ID = 32003

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE} | rank: {LORA_RANK} | batch: {BATCH_SIZE} | epochs: {EPOCHS}")
print(f"Data:   {DATA_PATH} (exists={DATA_PATH.exists()})")

In [ ]:
from huggingface_hub import hf_hub_download

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
for filename in ["model.safetensors", "tokenizer_spm_32k_3.model"]:
    dest = WEIGHTS_DIR / filename
    if dest.exists():
        print(f"Already present: {filename}")
        continue
    print(f"Downloading {filename} ...")
    hf_hub_download(repo_id=HF_PATCHED_REPO, filename=filename,
                    local_dir=str(WEIGHTS_DIR), token=HF_TOKEN)
    print(f"  ✓ {filename}")
print("Weights ready.")

In [ ]:
tok = spm.SentencePieceProcessor()
tok.Load(str(WEIGHTS_DIR / "tokenizer_spm_32k_3.model"))
assert tok.get_piece_size() == 32004, f"Expected 32004 tokens, got {tok.get_piece_size()}"
print(f"Tokenizer: {tok.get_piece_size()} tokens")
print("Special tokens:", [tok.id_to_piece(i) for i in [32000,32001,32002,32003]])

In [ ]:
torch.cuda.empty_cache(); gc.collect()
free, total = torch.cuda.mem_get_info()
print(f"GPU: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

print("Loading model to CPU then moving to GPU ...")
lm_model = get_moshi_lm(WEIGHTS_DIR / "model.safetensors", device="cpu", dtype=DTYPE)
lm_model = lm_model.to(DEVICE)
lm_model.train()
print(f"Loaded: {sum(p.numel() for p in lm_model.parameters())/1e9:.2f}B params  text_card={lm_model.text_card}")
free2, _ = torch.cuda.mem_get_info()
print(f"GPU after load: {free2/1e9:.1f} GB free")

In [ ]:
from peft import LoraConfig, get_peft_model

for p in lm_model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["out_proj", "in_proj"], bias="none",
)
lm_model = get_peft_model(lm_model, lora_config)
lm_model.print_trainable_parameters()

In [ ]:
class ToolCallDataset(Dataset):
    def __init__(self, path, max_len=MAX_SEQ_LEN):
        self.examples = []
        with open(path) as f:
            for line in f:
                ex = json.loads(line)
                t = torch.tensor(ex["tokens"][:max_len], dtype=torch.long)
                m = torch.tensor(ex["mask"][:max_len],   dtype=torch.long)
                self.examples.append({"tokens": t, "mask": m})
        print(f"Dataset: {len(self.examples)} examples")

    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

def collate(batch):
    L = max(b["tokens"].shape[0] for b in batch)
    tokens = torch.zeros(len(batch), L, dtype=torch.long)
    masks  = torch.zeros(len(batch), L, dtype=torch.long)
    for i, b in enumerate(batch):
        n = b["tokens"].shape[0]
        tokens[i,:n] = b["tokens"]
        masks[i,:n]  = b["mask"]
    return {"tokens": tokens, "masks": masks}

dataset    = ToolCallDataset(DATA_PATH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate, num_workers=0)
print(f"Batches/epoch: {len(dataloader)}")

In [ ]:
def compute_loss(model, tokens, masks):
    B, T = tokens.shape
    base = model.base_model.model if hasattr(model, 'base_model') else model
    codes = torch.zeros(B, 1 + base.n_q, T, dtype=torch.long, device=tokens.device)
    codes[:, 0] = tokens
    out = base.forward_train(codes)
    tl = out.text_logits[:, 0]   # [B, T, text_card]
    vm = out.text_mask[:, 0]     # [B, T]
    flat_m = (masks.bool() & vm).reshape(-1)
    if flat_m.sum() == 0:
        return torch.tensor(0., requires_grad=True, device=tokens.device)
    return nn.functional.cross_entropy(
        tl.reshape(-1, tl.shape[-1])[flat_m],
        tokens.reshape(-1)[flat_m]
    )

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in lm_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(dataloader))

lm_model.train()
global_step = 0
optimizer.zero_grad()

for epoch in range(EPOCHS):
    epoch_loss, n = 0., 0
    for step, batch in enumerate(dataloader):
        tokens = batch["tokens"].to(DEVICE)
        masks  = batch["masks"].to(DEVICE)
        loss   = compute_loss(lm_model, tokens, masks) / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM
        n += 1
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in lm_model.parameters() if p.requires_grad], 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            global_step += 1
            if global_step % 20 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS}  step {global_step}"
                      f"  loss {epoch_loss/n:.4f}"
                      f"  lr {scheduler.get_last_lr()[0]:.2e}")
    print(f"=== Epoch {epoch+1} done  loss={epoch_loss/n:.4f} ===")

print("Training complete.")

In [ ]:
from safetensors.torch import save_file

print("Merging LoRA weights ...")
merged = lm_model.merge_and_unload()
merged.eval()

raw_sd = merged.state_dict()
PREFIX = "base_model.model."
while any(k.startswith(PREFIX) for k in raw_sd):
    raw_sd = {(k[len(PREFIX):] if k.startswith(PREFIX) else k): v for k, v in raw_sd.items()}

out_weight = OUT_DIR / "model.safetensors"
save_file({k: v.contiguous() for k, v in raw_sd.items()}, str(out_weight))
shutil.copy2(WEIGHTS_DIR / "tokenizer_spm_32k_3.model",
             OUT_DIR / "tokenizer_spm_32k_3.model")
print(f"Saved → {OUT_DIR}")

In [ ]:
# Sanity check — load on CPU, use +1 dummy slot for correct next-token prediction
del merged, lm_model
torch.cuda.empty_cache(); gc.collect()

check = get_moshi_lm(OUT_DIR / "model.safetensors", device="cpu", dtype=torch.float32)
check.eval()

prompt_ids = tok.encode("what time is it")
ids = torch.tensor([prompt_ids], dtype=torch.long)
generated = []
with torch.no_grad():
    for _ in range(40):
        n_q = check.n_q
        # +1 dummy slot: text_logits[-1] predicts the token AFTER ids
        codes = torch.zeros(1, 1 + n_q, ids.shape[1] + 1, dtype=torch.long)
        codes[:, 0, :ids.shape[1]] = ids
        out  = check.forward_train(codes)
        nxt  = out.text_logits[0, 0, -1].argmax(-1).item()
        generated.append(nxt)
        ids  = torch.cat([ids, torch.tensor([[nxt]])], dim=1)
        if nxt == TOOL_END_ID:
            break

readable = []
for t in generated:
    if   t == TOOL_CALL_ID:        readable.append("<|tool_call|>")
    elif t == TOOL_END_ID:         readable.append("<|tool_end|>")
    elif t == PAD_ID:              readable.append("[PAD]")
    elif t < tok.get_piece_size(): readable.append(tok.id_to_piece(t))
print("Generated:", "".join(readable))

# Also show top-5 after trigger for diagnosis
with torch.no_grad():
    codes_dbg = torch.zeros(1, 1 + check.n_q, len(prompt_ids) + 1, dtype=torch.long)
    codes_dbg[:, 0, :len(prompt_ids)] = torch.tensor([prompt_ids])
    out_dbg = check.forward_train(codes_dbg)
    top5_vals, top5_ids = out_dbg.text_logits[0, 0, -1].topk(5)
    print("\nTop-5 after 'what time is it':")
    for v, i in zip(top5_vals.tolist(), top5_ids.tolist()):
        piece = f"<special_{i}>" if i >= 32000 else tok.id_to_piece(i)
        print(f"  {i:6d}  {piece:20s}  {v:.3f}")

if TOOL_CALL_ID in generated:
    print("\n✓ Fine-tuning worked!")
else:
    print("\n⚠ Tool tokens not emitted — check top-5 above")

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(HF_OUTPUT_REPO, private=True, exist_ok=True, token=HF_TOKEN)
print(f"Repo: https://huggingface.co/{HF_OUTPUT_REPO}")

for f in OUT_DIR.iterdir():
    if not f.is_file():
        continue
    print(f"Uploading {f.name}  ({f.stat().st_size/1e9:.2f} GB) ...")
    api.upload_file(path_or_fileobj=str(f), path_in_repo=f.name,
                    repo_id=HF_OUTPUT_REPO, token=HF_TOKEN)
    print(f"  ✓ {f.name}")

print(f'''
Upload complete!

On your server:
  bash run_server.sh --hf-model {HF_OUTPUT_REPO}
''')